# Skagit Parcel — ArcGIS Geo Backfill

Discovers all Skagit PropertyMap MapServer layers, queries each one by parcel ID in bulk,
and writes results into two Cloudflare D1 tables:

- **`parcel_geo_layers`** — full attribute payload per layer (JSON blob)
- **`parcel_cards`** — patches `latitude`, `longitude`, `geometry` from layer 5 (polygons)

Run all cells top-to-bottom. Safe to re-run — uses `INSERT OR REPLACE`.
Set `ONLY_MISSING = True` to skip parcels already enriched.

In [ ]:
# No extra installs needed — requests is built into Colab
import requests, json, time, math
from getpass import getpass
from datetime import date
from IPython.display import display, HTML
print('imports ok')

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
# Paste your Cloudflare account ID here (not secret)
CF_ACCOUNT_ID  = '55426a025182255079a703166b6cce8a'
D1_DATABASE_ID = 'bd1fd2cb-9d82-4068-a79a-de55c83cc981'

# Token is prompted so it doesn't appear in notebook history
CF_API_TOKEN = getpass('CF API token: ')

# ── Options ───────────────────────────────────────────────────────────────────
ONLY_MISSING = False   # True = skip parcels already in parcel_geo_layers
BATCH_SIZE   = 50      # IDs per ArcGIS request (keep ≤100)
SLEEP_SEC    = 0.2     # delay between ArcGIS requests (~5 req/sec)
LIMIT        = None    # set an int to cap total parcels (useful for testing)

MAPSERVER    = 'https://gis.skagitcountywa.gov/arcgis/rest/services/Assessor/PropertyMap/MapServer'
GEO_LAYER_ID = 5       # polygon layer → lat/lon/geometry in parcel_cards
TODAY        = date.today().isoformat()

CF_BASE    = f'https://api.cloudflare.com/client/v4/accounts/{CF_ACCOUNT_ID}'
AUTH       = {'Authorization': f'Bearer {CF_API_TOKEN}'}
print(f'config ok — date={TODAY}')

In [ ]:
# ── D1 helpers ────────────────────────────────────────────────────────────────

def d1_query(sql, params=None):
    """Run a single SQL statement, return list of row dicts."""
    body = [{'sql': sql, 'params': params or []}]
    r = requests.post(
        f'{CF_BASE}/d1/database/{D1_DATABASE_ID}/query',
        headers={**AUTH, 'Content-Type': 'application/json'},
        json=body, timeout=30,
    )
    r.raise_for_status()
    data = r.json()
    if not data.get('success'):
        raise RuntimeError(f"D1 error: {data.get('errors')}")
    return data['result'][0]['results']


def d1_batch(statements):
    """Run a list of {sql, params} dicts in one D1 request."""
    r = requests.post(
        f'{CF_BASE}/d1/database/{D1_DATABASE_ID}/query',
        headers={**AUTH, 'Content-Type': 'application/json'},
        json=statements, timeout=60,
    )
    r.raise_for_status()
    data = r.json()
    if not data.get('success'):
        raise RuntimeError(f"D1 batch error: {data.get('errors')}")
    return data['result']


def d1_all(sql):
    """Page through a SELECT in 10k-row chunks, return all rows."""
    rows, offset, page = [], 0, 10_000
    while True:
        chunk = d1_query(f'{sql} LIMIT {page} OFFSET {offset}')
        rows.extend(chunk)
        if len(chunk) < page:
            break
        offset += page
    return rows


# ── ArcGIS helpers ────────────────────────────────────────────────────────────

def discover_layers():
    r = requests.get(f'{MAPSERVER}?f=json', timeout=15)
    r.raise_for_status()
    return [{'id': l['id'], 'name': l['name']} for l in r.json().get('layers', [])]


def query_layer(layer_id, parcel_ids):
    in_list = ','.join(f"'{pid.replace(chr(39), chr(39)*2)}'" for pid in parcel_ids)
    params = {
        'where':          f'PARCELID IN ({in_list})',
        'outFields':      '*',
        'returnGeometry': 'true' if layer_id == GEO_LAYER_ID else 'false',
        'outSR':          '4326',
        'f':              'json',
    }
    r = requests.get(f'{MAPSERVER}/{layer_id}/query', params=params, timeout=30)
    r.raise_for_status()
    return r.json()


def centroid(rings):
    coords = rings[0]
    lons   = [c[0] for c in coords]
    lats   = [c[1] for c in coords]
    return (min(lats) + max(lats)) / 2, (min(lons) + max(lons)) / 2


print('helpers defined')

In [ ]:
# ── Step 1: Discover layers ───────────────────────────────────────────────────
layers = discover_layers()
print(f'Found {len(layers)} layers:')
for l in layers:
    print(f'  [{l["id"]}] {l["name"]}')

In [ ]:
# ── Step 2: Load parcel IDs from D1 ──────────────────────────────────────────
if ONLY_MISSING:
    sql = '''SELECT parcel_id FROM parcel_cards
             WHERE parcel_id NOT IN (SELECT parcel_id FROM parcel_geo_layers)
             ORDER BY parcel_id'''
else:
    sql = 'SELECT parcel_id FROM parcel_cards ORDER BY parcel_id'

print('Loading parcel IDs...')
rows    = d1_all(sql)
all_ids = [r['parcel_id'] for r in rows]
if LIMIT:
    all_ids = all_ids[:LIMIT]

print(f'{len(all_ids):,} parcels to process  (only_missing={ONLY_MISSING})')

In [ ]:
# ── Step 3: Query each layer in batches ──────────────────────────────────────
# enriched[parcel_id] = { 'LayerName': {attrs}, ..., '_geo': {lat,lon,geometry} }
enriched = {}

total_batches = math.ceil(len(all_ids) / BATCH_SIZE)

for layer in layers:
    lid, lname = layer['id'], layer['name']
    hits = 0
    errors = 0

    for batch_num, i in enumerate(range(0, len(all_ids), BATCH_SIZE)):
        chunk = all_ids[i:i + BATCH_SIZE]

        try:
            data = query_layer(lid, chunk)
        except Exception as e:
            print(f'  layer {lid} batch {batch_num} error: {e}')
            errors += 1
            time.sleep(2)
            continue

        for feature in data.get('features', []):
            pid = feature.get('attributes', {}).get('PARCELID')
            if not pid:
                continue
            if pid not in enriched:
                enriched[pid] = {}

            attrs = {k: v for k, v in feature.get('attributes', {}).items() if v is not None}
            enriched[pid][lname] = attrs

            if lid == GEO_LAYER_ID:
                rings = feature.get('geometry', {}).get('rings')
                if rings:
                    lat, lon = centroid(rings)
                    enriched[pid]['_geo'] = {
                        'latitude':  lat,
                        'longitude': lon,
                        'geometry':  json.dumps({'type': 'Polygon', 'coordinates': rings}),
                    }
            hits += 1

        if batch_num % 50 == 0:
            pct = (i + len(chunk)) / len(all_ids) * 100
            print(f'  layer [{lid}] {lname}: {pct:.0f}%  ({hits} hits so far)')

        time.sleep(SLEEP_SEC)

    print(f'layer [{lid}] {lname} done — {hits} features, {errors} errors')

print(f'\nTotal parcels enriched: {len(enriched):,}')

In [ ]:
# ── Step 4: Write to D1 ───────────────────────────────────────────────────────
GEO_LAYERS_SQL = '''INSERT OR REPLACE INTO parcel_geo_layers
  (parcel_id, enriched_date, layers) VALUES (?, ?, ?)'''

PATCH_GEO_SQL = '''UPDATE parcel_cards
  SET latitude=?, longitude=?, geometry=? WHERE parcel_id=?'''

enriched_ids = list(enriched.keys())
WRITE_BATCH  = 100
layers_writ  = 0
geo_patched  = 0

for i in range(0, len(enriched_ids), WRITE_BATCH):
    chunk      = enriched_ids[i:i + WRITE_BATCH]
    statements = []

    for pid in chunk:
        data = enriched[pid]
        geo  = data.get('_geo')

        layers_blob = json.dumps({k: v for k, v in data.items() if k != '_geo'})
        statements.append({'sql': GEO_LAYERS_SQL, 'params': [pid, TODAY, layers_blob]})
        layers_writ += 1

        if geo:
            statements.append({'sql': PATCH_GEO_SQL,
                                'params': [geo['latitude'], geo['longitude'], geo['geometry'], pid]})
            geo_patched += 1

    d1_batch(statements)

    if (i // WRITE_BATCH) % 10 == 0:
        pct = min(i + WRITE_BATCH, len(enriched_ids)) / len(enriched_ids) * 100
        print(f'  writing: {pct:.0f}%  ({layers_writ} rows written)')

print(f'\nDone — {layers_writ:,} parcel_geo_layers rows, {geo_patched:,} parcel_cards patched')

In [ ]:
# ── Step 5: Quick verification ────────────────────────────────────────────────
total_cards  = d1_query('SELECT COUNT(*) as n FROM parcel_cards')[0]['n']
total_layers = d1_query('SELECT COUNT(*) as n FROM parcel_geo_layers')[0]['n']
has_geo      = d1_query('SELECT COUNT(*) as n FROM parcel_cards WHERE latitude IS NOT NULL')[0]['n']
no_geo       = d1_query('SELECT COUNT(*) as n FROM parcel_cards WHERE latitude IS NULL')[0]['n']

print(f'parcel_cards:      {total_cards:,} total  |  {has_geo:,} with lat/lon  |  {no_geo:,} still missing')
print(f'parcel_geo_layers: {total_layers:,} rows')

# Sample one enriched parcel
sample = d1_query('SELECT parcel_id, enriched_date, layers FROM parcel_geo_layers LIMIT 1')
if sample:
    pid = sample[0]['parcel_id']
    layers_data = json.loads(sample[0]['layers'])
    print(f'\nSample parcel: {pid}')
    for lname, attrs in layers_data.items():
        print(f'  {lname}: {list(attrs.keys())[:6]}')